# 🧠 Système Multi-Agent D-A — Grok 4.2 Autonome

Ce notebook Colab crée un système complet:
1. **Création d'emails réels** (mail.com/gmx.com via Playwright)
2. **Inscription Grok** avec ces emails
3. **Sessions multiples Grok 4.2** en parallèle
4. **Système multi-agents** : Générateur + Critique + Vérificateur + Gardien mémoire
5. **Sauvegarde continue** sur Google Drive

⚠️ Exécuter les cellules dans l'ordre. La première installation prend ~2 min.

In [ ]:
# === CELLULE 1: Installation ===
!pip install -q playwright asyncio nest_asyncio
!playwright install chromium
!playwright install-deps
!pip install -q google-colab
print('✅ Installation complète')

In [ ]:
# === CELLULE 2: Monter Google Drive pour sauvegarde persistante ===
from google.colab import drive
import os

drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/DA_MultiAgent'
os.makedirs(f'{SAVE_DIR}/rounds', exist_ok=True)
os.makedirs(f'{SAVE_DIR}/emails', exist_ok=True)
os.makedirs(f'{SAVE_DIR}/sessions', exist_ok=True)
os.makedirs(f'{SAVE_DIR}/logs', exist_ok=True)
print(f'✅ Google Drive monté. Sauvegarde dans: {SAVE_DIR}')

In [ ]:
# === CELLULE 3: Création d'emails réels via mail.com ===
import asyncio
import nest_asyncio
import json
import random
import string
import time
from datetime import datetime
from playwright.async_api import async_playwright

nest_asyncio.apply()

EMAILS_FILE = f'{SAVE_DIR}/emails/accounts.json'

def load_accounts():
    if os.path.exists(EMAILS_FILE):
        return json.loads(open(EMAILS_FILE).read())
    return []

def save_accounts(accounts):
    with open(EMAILS_FILE, 'w') as f:
        f.write(json.dumps(accounts, indent=2))

def gen_name():
    prefixes = ['alex','sam','jordan','casey','morgan','taylor','riley','avery','quinn','drew']
    suffixes = ['research','math','dev','lab','sci','calc','num','pde','fem','algo']
    return random.choice(prefixes) + random.choice(suffixes) + str(random.randint(100,999))

async def create_mail_com_account():
    """Crée un compte mail.com réel via Playwright headless."""
    username = gen_name()
    password = 'DA_' + ''.join(random.choices(string.ascii_letters + string.digits, k=12)) + '!'
    
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(
            viewport={'width': 1280, 'height': 720},
            user_agent='Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
        )
        page = await context.new_page()
        
        try:
            # Aller sur mail.com signup
            await page.goto('https://signup.mail.com/premium/us/', timeout=30000)
            await page.wait_for_load_state('networkidle', timeout=15000)
            
            # Chercher le lien Free signup
            free_link = page.locator('text=Sign up for free')
            if await free_link.count() > 0:
                await free_link.first.click()
                await page.wait_for_load_state('networkidle', timeout=15000)
            
            # Remplir le formulaire d'inscription
            # Prénom
            fname_field = page.locator('input[name="firstName"], #firstName')
            if await fname_field.count() > 0:
                await fname_field.fill(username[:4].capitalize())
            
            # Nom
            lname_field = page.locator('input[name="lastName"], #lastName')
            if await lname_field.count() > 0:
                await lname_field.fill(username[4:].capitalize())
            
            # Email souhaité
            email_field = page.locator('input[name="emailAddress"], input[name="desiredEmailAddress"], #desiredEmailAddress')
            if await email_field.count() > 0:
                await email_field.fill(username)
            
            # Password
            pass_field = page.locator('input[name="password"], input[type="password"]')
            if await pass_field.count() > 0:
                await pass_field.first.fill(password)
            
            # Date de naissance
            dob_fields = page.locator('select[name="birthMonth"], select[name="birthDay"], select[name="birthYear"]')
            if await dob_fields.count() >= 3:
                await page.select_option('select[name="birthMonth"]', str(random.randint(1,12)))
                await page.select_option('select[name="birthDay"]', str(random.randint(1,28)))
                await page.select_option('select[name="birthYear"]', str(random.randint(1985,2000)))
            
            # Genre
            gender = page.locator('select[name="gender"]')
            if await gender.count() > 0:
                await page.select_option('select[name="gender"]', 'male')
            
            # Screenshot pour debug
            await page.screenshot(path=f'/content/signup_{username}.png')
            
            # Soumettre
            submit = page.locator('button[type="submit"], input[type="submit"]')
            if await submit.count() > 0:
                await submit.first.click()
                await page.wait_for_load_state('networkidle', timeout=30000)
            
            await page.screenshot(path=f'/content/result_{username}.png')
            
            # Vérifier succès
            content = await page.content()
            success = 'welcome' in content.lower() or 'inbox' in content.lower() or 'success' in content.lower()
            
            account = {
                'email': f'{username}@mail.com',
                'password': password,
                'created': datetime.now().isoformat(),
                'status': 'created' if success else 'pending_verification',
                'grok_registered': False,
                'screenshots': [f'signup_{username}.png', f'result_{username}.png']
            }
            
            print(f"{'✅' if success else '⚠️'} {account['email']} — {account['status']}")
            return account
            
        except Exception as e:
            print(f'❌ Erreur création {username}: {e}')
            await page.screenshot(path=f'/content/error_{username}.png')
            return {'email': f'{username}@mail.com', 'password': password, 'status': f'error: {e}', 'grok_registered': False}
        finally:
            await browser.close()

# === Créer N comptes ===
NUM_ACCOUNTS = 4  # 4 agents = 4 comptes

async def create_all_accounts():
    accounts = load_accounts()
    existing = len(accounts)
    needed = max(0, NUM_ACCOUNTS - existing)
    
    if needed == 0:
        print(f'✅ {existing} comptes déjà créés')
        for a in accounts:
            print(f"  {a['email']} — grok: {a.get('grok_registered', False)}")
        return accounts
    
    print(f'Création de {needed} comptes email...')
    for i in range(needed):
        print(f'\n--- Compte {existing + i + 1}/{NUM_ACCOUNTS} ---')
        account = await create_mail_com_account()
        accounts.append(account)
        save_accounts(accounts)
        await asyncio.sleep(5)  # Anti rate-limit
    
    return accounts

accounts = asyncio.get_event_loop().run_until_complete(create_all_accounts())
print(f'\n📧 {len(accounts)} comptes disponibles')

In [ ]:
# === CELLULE 4: Inscription Grok avec les emails créés ===

async def register_grok(account):
    """Inscrit un email sur grok.com"""
    if account.get('grok_registered'):
        print(f"✅ {account['email']} déjà inscrit sur Grok")
        return account
    
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(
            viewport={'width': 1280, 'height': 720},
            user_agent='Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36'
        )
        page = await context.new_page()
        
        try:
            await page.goto('https://grok.com', timeout=30000)
            await page.wait_for_load_state('networkidle', timeout=15000)
            
            # Chercher Sign up / Log in
            signup = page.locator('text=Sign up, text=Log in, text=Get started, a[href*="auth"], button:has-text("Sign")').first
            if await signup.count() > 0:
                await signup.click()
                await page.wait_for_load_state('networkidle', timeout=15000)
            
            # Chercher option email (pas Google/Apple)
            email_option = page.locator('text=email, text=Email, input[type="email"], text=Use email').first
            if await email_option.count() > 0:
                await email_option.click()
                await asyncio.sleep(2)
            
            # Remplir email
            email_input = page.locator('input[type="email"], input[name="email"]').first
            if await email_input.count() > 0:
                await email_input.fill(account['email'])
            
            await page.screenshot(path=f"/content/grok_signup_{account['email'].split('@')[0]}.png")
            
            # Soumettre
            submit = page.locator('button[type="submit"], button:has-text("Continue"), button:has-text("Next")').first
            if await submit.count() > 0:
                await submit.click()
                await page.wait_for_load_state('networkidle', timeout=15000)
            
            await page.screenshot(path=f"/content/grok_result_{account['email'].split('@')[0]}.png")
            
            content = await page.content()
            if 'verification' in content.lower() or 'code' in content.lower() or 'check your email' in content.lower():
                account['grok_status'] = 'verification_code_sent'
                print(f"📨 {account['email']} — code de vérification envoyé")
            elif 'grok' in content.lower() and ('chat' in content.lower() or 'ask' in content.lower()):
                account['grok_registered'] = True
                account['grok_status'] = 'active'
                print(f"✅ {account['email']} — Grok actif !")
            else:
                account['grok_status'] = 'unknown'
                print(f"⚠️ {account['email']} — statut inconnu, voir screenshot")
            
            return account
            
        except Exception as e:
            print(f"❌ {account['email']} — erreur: {e}")
            account['grok_status'] = f'error: {e}'
            return account
        finally:
            await browser.close()

async def register_all_grok():
    accounts = load_accounts()
    for i, account in enumerate(accounts):
        print(f'\n--- Grok inscription {i+1}/{len(accounts)} ---')
        accounts[i] = await register_grok(account)
        save_accounts(accounts)
        await asyncio.sleep(5)
    return accounts

accounts = asyncio.get_event_loop().run_until_complete(register_all_grok())

In [ ]:
# === CELLULE 5: Vérification email — récupérer les codes Grok ===

async def get_verification_code(account):
    """Se connecte à mail.com et récupère le code de vérification Grok."""
    if account.get('grok_registered'):
        return account
    
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context()
        page = await context.new_page()
        
        try:
            # Login mail.com
            await page.goto('https://www.mail.com/login', timeout=30000)
            await page.wait_for_load_state('networkidle')
            
            email_input = page.locator('input[name="username"], input[id="username"]').first
            if await email_input.count() > 0:
                await email_input.fill(account['email'])
            
            pass_input = page.locator('input[name="password"], input[type="password"]').first
            if await pass_input.count() > 0:
                await pass_input.fill(account['password'])
            
            submit = page.locator('button[type="submit"]').first
            if await submit.count() > 0:
                await submit.click()
                await page.wait_for_load_state('networkidle', timeout=15000)
            
            await asyncio.sleep(3)
            
            # Chercher email de Grok/xAI
            grok_email = page.locator('text=grok, text=xAI, text=x.ai, text=verification').first
            if await grok_email.count() > 0:
                await grok_email.click()
                await asyncio.sleep(2)
                
                # Extraire le code
                content = await page.content()
                import re
                codes = re.findall(r'\b(\d{6})\b', content)
                if codes:
                    account['verification_code'] = codes[0]
                    print(f"🔑 {account['email']} — code: {codes[0]}")
                else:
                    # Chercher un lien de vérification
                    links = re.findall(r'https?://[^\s"<>]+(?:verify|confirm|auth)[^\s"<>]*', content)
                    if links:
                        account['verification_link'] = links[0]
                        print(f"🔗 {account['email']} — lien: {links[0][:60]}...")
            else:
                print(f"📭 {account['email']} — pas d'email Grok trouvé")
                await page.screenshot(path=f"/content/inbox_{account['email'].split('@')[0]}.png")
            
            return account
        except Exception as e:
            print(f"❌ {account['email']} — erreur: {e}")
            return account
        finally:
            await browser.close()

async def verify_all():
    accounts = load_accounts()
    for i, acc in enumerate(accounts):
        if acc.get('grok_status') == 'verification_code_sent':
            accounts[i] = await get_verification_code(acc)
            save_accounts(accounts)
            await asyncio.sleep(3)
    return accounts

accounts = asyncio.get_event_loop().run_until_complete(verify_all())

In [ ]:
# === CELLULE 6: Compléter vérification Grok avec le code ===

async def complete_grok_verification(account):
    """Entre le code de vérification sur Grok."""
    code = account.get('verification_code')
    link = account.get('verification_link')
    if not code and not link:
        return account
    if account.get('grok_registered'):
        return account
    
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        # Sauvegarder le storage state pour réutiliser la session
        context = await browser.new_context()
        page = await context.new_page()
        
        try:
            if link:
                await page.goto(link, timeout=30000)
            else:
                await page.goto('https://grok.com', timeout=30000)
                await page.wait_for_load_state('networkidle')
                # Trouver et remplir le champ code
                code_input = page.locator('input[name="code"], input[type="text"], input[placeholder*="code"]').first
                if await code_input.count() > 0:
                    await code_input.fill(code)
                    submit = page.locator('button[type="submit"], button:has-text("Verify")').first
                    if await submit.count() > 0:
                        await submit.click()
            
            await page.wait_for_load_state('networkidle', timeout=15000)
            await asyncio.sleep(3)
            
            # Sauvegarder la session pour réutilisation
            storage = await context.storage_state()
            session_file = f"{SAVE_DIR}/sessions/{account['email'].split('@')[0]}_session.json"
            with open(session_file, 'w') as f:
                json.dump(storage, f)
            
            content = await page.content()
            if 'grok' in content.lower() and ('ask' in content.lower() or 'chat' in content.lower() or 'message' in content.lower()):
                account['grok_registered'] = True
                account['grok_status'] = 'active'
                account['session_file'] = session_file
                print(f"✅ {account['email']} — Grok ACTIF !")
            else:
                await page.screenshot(path=f"/content/grok_verify_{account['email'].split('@')[0]}.png")
                print(f"⚠️ {account['email']} — vérifier screenshot")
            
            return account
        except Exception as e:
            print(f"❌ {account['email']} — {e}")
            return account
        finally:
            await browser.close()

async def complete_all_verification():
    accounts = load_accounts()
    for i, acc in enumerate(accounts):
        if not acc.get('grok_registered') and (acc.get('verification_code') or acc.get('verification_link')):
            accounts[i] = await complete_grok_verification(acc)
            save_accounts(accounts)
    return accounts

accounts = asyncio.get_event_loop().run_until_complete(complete_all_verification())
active = [a for a in accounts if a.get('grok_registered')]
print(f'\n🎯 {len(active)}/{len(accounts)} comptes Grok actifs')

In [ ]:
# === CELLULE 7: Classe GrokSession — gère une session Grok 4.2 ===

class GrokSession:
    """Gère une session browser persistante vers Grok 4.2."""
    
    def __init__(self, account, role, playwright_instance):
        self.account = account
        self.role = role  # 'generator', 'critic', 'verifier', 'memory_guard'
        self.pw = playwright_instance
        self.browser = None
        self.page = None
        self.context = None
        self.history = []
    
    async def connect(self):
        """Ouvre le browser et restaure la session Grok."""
        self.browser = await self.pw.chromium.launch(headless=True)
        
        # Restaurer la session si disponible
        session_file = self.account.get('session_file')
        if session_file and os.path.exists(session_file):
            storage = json.loads(open(session_file).read())
            self.context = await self.browser.new_context(storage_state=storage)
        else:
            self.context = await self.browser.new_context()
        
        self.page = await self.context.new_page()
        await self.page.goto('https://grok.com', timeout=30000)
        await self.page.wait_for_load_state('networkidle', timeout=15000)
        
        # Sélectionner Grok 4.2 si disponible
        model_selector = self.page.locator('text=Grok 4, text=grok-4, button:has-text("4")')
        if await model_selector.count() > 0:
            await model_selector.first.click()
            await asyncio.sleep(1)
        
        print(f"🔌 {self.role} ({self.account['email']}) — connecté")
    
    async def send_message(self, message):
        """Envoie un message à Grok et attend la réponse."""
        try:
            # Trouver le textarea
            textarea = self.page.locator('textarea, div[contenteditable="true"], .tiptap.ProseMirror').first
            if await textarea.count() == 0:
                return 'ERREUR: textarea non trouvé'
            
            # Vider et remplir
            await textarea.click()
            await textarea.fill(message[:10000])  # Grok a une limite
            await asyncio.sleep(0.5)
            
            # Envoyer (Enter ou bouton)
            send_btn = self.page.locator('button[aria-label="Send"], button:has-text("Send"), button[type="submit"]')
            if await send_btn.count() > 0:
                await send_btn.first.click()
            else:
                await textarea.press('Enter')
            
            # Attendre la réponse (polling jusqu'à ce que Grok arrête de taper)
            await asyncio.sleep(5)  # Attendre le début de la réponse
            
            last_content = ''
            stable_count = 0
            max_wait = 120  # 2 min max
            elapsed = 0
            
            while stable_count < 3 and elapsed < max_wait:
                await asyncio.sleep(3)
                elapsed += 3
                
                # Récupérer le dernier message de Grok
                messages = self.page.locator('[class*="message"], [class*="response"], [data-role="assistant"]')
                count = await messages.count()
                if count > 0:
                    current = await messages.last.inner_text()
                    if current == last_content:
                        stable_count += 1
                    else:
                        stable_count = 0
                        last_content = current
            
            self.history.append({'role': 'user', 'content': message})
            self.history.append({'role': 'grok', 'content': last_content})
            
            return last_content
            
        except Exception as e:
            return f'ERREUR: {e}'
    
    async def close(self):
        if self.context:
            storage = await self.context.storage_state()
            session_file = f"{SAVE_DIR}/sessions/{self.account['email'].split('@')[0]}_session.json"
            with open(session_file, 'w') as f:
                json.dump(storage, f)
        if self.browser:
            await self.browser.close()

print('✅ GrokSession défini')

In [ ]:
# === CELLULE 8: Orchestrateur Multi-Agent Autonome ===

EXIGENCES = """
EXIGENCES OBLIGATOIRES:
- Solutions PLUS PERFORMANTES que les meilleures existantes
- Résoudre des problèmes ACTUELLEMENT NON RÉSOLUS
- Code Python EXÉCUTABLE (pas de pseudocode)
- EXÉCUTER le code et rapporter les VRAIS résultats
- Ratios >2× intéressant, >5× publiable, >10× significatif
- Comparer avec l'état de l'art (citer la référence)
- Vérifier C = DA-AD non-trivial
- SAUVEGARDER tout
- Répondre en FRANÇAIS
"""

ROLES = {
    'generator': f"""Tu es un mathématicien-chercheur. Tu génères des solutions ORIGINALES en théorie Discrétisation-Approximation (D-A).
Le commutateur C = DA - AD entre discrétisation et approximation est ton outil principal.
ATTENTION: C peut être trivial (matrice nulle si DA=I). Toujours vérifier.
{EXIGENCES}""",

    'critic': f"""Tu es un critique mathématique IMPITOYABLE. Tu vérifies CHAQUE claim.
- EXÉCUTE le code indépendamment
- Vérifie les FONDATIONS (C non-trivial? Borne théorique?)
- Compare avec l'état de l'art RÉEL
- Score 0-10. Si < 7, REJETÉ.
- Détecte les hallucinations (chiffres inventés, ratios inversés)
Erreur passée: confirmer que le code tourne sans vérifier que les maths sont correctes. C_coarse était NULLE pendant 10 rounds.
{EXIGENCES}""",

    'verifier': f"""Tu es un vérificateur INDÉPENDANT. Tu n'as PAS accès aux critiques.
Tu reçois uniquement le code et les résultats bruts.
- Exécute chaque code de zéro
- Vérifie les fondations mathématiques
- Cherche si la méthode existe déjà sous un autre nom
- Cherche les patterns d'erreur systématiques
{EXIGENCES}""",

    'memory_guard': """Tu es le gardien de mémoire du système multi-agents.
- Vérifie que TOUS les rounds sont sauvegardés (pas de trous)
- Maintiens un index complet
- Détecte les incohérences (scores, fichiers manquants)
- ALERTE si quelque chose manque
Réponds en FRANÇAIS."""
}

async def run_multi_agent_system():
    """Boucle principale du système multi-agents Grok."""
    accounts = load_accounts()
    active = [a for a in accounts if a.get('grok_registered')]
    
    if len(active) < 2:
        print('❌ Pas assez de comptes Grok actifs. Exécuter les cellules 3-6 d\'abord.')
        return
    
    # Assigner les rôles
    role_names = ['generator', 'critic', 'verifier', 'memory_guard']
    assignments = {}
    
    async with async_playwright() as pw:
        sessions = {}
        for i, role in enumerate(role_names[:len(active)]):
            session = GrokSession(active[i], role, pw)
            await session.connect()
            sessions[role] = session
            assignments[role] = active[i]['email']
        
        print(f'\n🎯 Système lancé avec {len(sessions)} agents:')
        for role, email in assignments.items():
            print(f'  {role}: {email}')
        
        # Charger état
        state_file = f'{SAVE_DIR}/state.json'
        if os.path.exists(state_file):
            state = json.loads(open(state_file).read())
        else:
            state = {'round': 1, 'scores': [], 'corrections': 0}
        
        # === BOUCLE PRINCIPALE ===
        while True:
            n = state['round']
            ts = datetime.now().strftime('%H:%M:%S')
            print(f'\n{"="*60}')
            print(f'ROUND {n} — {ts}')
            print(f'{"="*60}')
            
            # Contexte des rounds précédents
            history = ''
            for i in range(max(1, n-3), n):
                vf = f'{SAVE_DIR}/rounds/round_{i}.md'
                if os.path.exists(vf):
                    history += open(vf).read()[:2000] + '\n'
            
            # --- PHASE 1: Génération ---
            gen_prompt = f"""Round {n}. {ROLES['generator']}

Historique récent:\n{history[:3000]}

Propose UNE contribution D-A avec:
1. Code Python complet exécutable
2. Résultats d'exécution réels
3. Comparaison état de l'art chiffrée
4. Ce qui est NOUVEAU"""
            
            print(f'  📐 Générateur...')
            gen_response = await sessions['generator'].send_message(gen_prompt)
            print(f'  ✓ Générateur: {len(gen_response)} chars')
            
            # --- PHASE 2: Critique (boucle de correction) ---
            max_attempts = 3
            attempt = 0
            score = 0
            critic_response = ''
            
            while attempt < max_attempts and score < 7:
                attempt += 1
                
                critic_prompt = f"""Vérifie ce Round {n} (tentative {attempt}).

{ROLES['critic']}

RÉPONSE DU GÉNÉRATEUR:
{gen_response[:6000]}

Score de 0 à 10. Si < 7, explique PRÉCISÉMENT quoi corriger."""
                
                print(f'  🔍 Critique (tentative {attempt})...')
                critic_response = await sessions['critic'].send_message(critic_prompt)
                
                import re
                score_match = re.search(r'[Ss]core\s*:?\s*(\d+(?:\.\d+)?)\s*/\s*10', critic_response)
                score = float(score_match.group(1)) if score_match else 0
                print(f'  📊 Score: {score}/10')
                
                if score < 7 and attempt < max_attempts:
                    state['corrections'] += 1
                    # Renvoyer la critique au générateur pour correction
                    fix_prompt = f"""Score {score}/10 — CORRIGE.

CRITIQUE:
{critic_response[:4000]}

Corrige les erreurs et repropose. Score minimum: 7/10."""
                    print(f'  🔄 Correction...')
                    gen_response = await sessions['generator'].send_message(fix_prompt)
            
            # --- PHASE 3: Vérificateur indépendant (tous les 10 rounds) ---
            verifier_response = ''
            if n % 10 == 0 and 'verifier' in sessions:
                verif_prompt = f"""Vérification indépendante rounds {max(1,n-9)} à {n}.
{ROLES['verifier']}
Code et résultats du dernier round:\n{gen_response[:5000]}"""
                print(f'  🔬 Vérificateur indépendant...')
                verifier_response = await sessions['verifier'].send_message(verif_prompt)
            
            # --- PHASE 4: Gardien mémoire ---
            if 'memory_guard' in sessions:
                mem_prompt = f"""Vérifie round {n}. Tous les fichiers round_1 à round_{n} doivent exister.
Score: {score}/10. Tentatives: {attempt}.
Historique scores: {state['scores']}"""
                await sessions['memory_guard'].send_message(mem_prompt)
            
            # --- SAUVEGARDE ---
            round_content = f"""# Round {n} — Score: {score}/10 — Tentatives: {attempt}
Date: {datetime.now().isoformat()}

## Générateur
{gen_response}

## Critique
{critic_response}

## Vérificateur Indépendant
{verifier_response if verifier_response else 'N/A (tous les 10 rounds)'}
"""
            with open(f'{SAVE_DIR}/rounds/round_{n}.md', 'w') as f:
                f.write(round_content)
            
            state['scores'].append(score)
            state['round'] = n + 1
            with open(state_file, 'w') as f:
                json.dump(state, f, indent=2)
            
            print(f'  ✅ Round {n} COMPLET — Score: {score}/10')
            print(f'  📈 Scores: {state["scores"]}')
            print(f'  💾 Sauvé: {SAVE_DIR}/rounds/round_{n}.md')
        
        # Cleanup
        for s in sessions.values():
            await s.close()

print('✅ Orchestrateur défini. Exécuter la cellule suivante pour lancer.')

In [ ]:
# === CELLULE 9: LANCER LE SYSTÈME ===
# Cette cellule tourne en CONTINU jusqu'à interruption

asyncio.get_event_loop().run_until_complete(run_multi_agent_system())

In [ ]:
# === CELLULE 10: Anti-déconnexion Colab ===
# Exécuter dans un onglet séparé pour garder Colab actif

import IPython
from google.colab import output

# Injecter du JS qui clique périodiquement pour éviter la déconnexion
display(IPython.display.Javascript('''
function keepAlive() {
    console.log("Keep alive: " + new Date().toISOString());
    // Simuler activité
    document.querySelector("colab-toolbar-button#connect")?.click();
}
setInterval(keepAlive, 60000);  // Toutes les minutes
console.log("Anti-déconnexion activé");
'''))
print('✅ Anti-déconnexion Colab activé')

In [ ]:
# === CELLULE 11: Rapport de statut (exécuter à tout moment) ===

state_file = f'{SAVE_DIR}/state.json'
if os.path.exists(state_file):
    state = json.loads(open(state_file).read())
    print(f"Round actuel: {state['round']}")
    print(f"Scores: {state['scores']}")
    print(f"Corrections: {state.get('corrections', 0)}")
    print(f"Score moyen: {sum(state['scores'])/len(state['scores']):.1f}/10" if state['scores'] else 'Aucun round')
    print(f"\nRounds sauvegardés:")
    import glob
    for f in sorted(glob.glob(f'{SAVE_DIR}/rounds/round_*.md')):
        size = os.path.getsize(f)
        print(f"  {os.path.basename(f)}: {size:,} bytes")
else:
    print('Système pas encore lancé')